# Milestone 5 — Cross-dataset synthesis & cross-validation

Combines the per-dataset aggregated tables from notebooks 01–04 (Allen scRNA, Allen
MERFISH, Vizgen, Zhuang) into a single **evidence table** keyed by (cell_type,
brain_area, gene), with detection rates, cross-dataset concordance, and a
**confidence tier** per row. Level-agnostic: works at whatever `cell_type_level`
the config uses.

Independence: Allen MERFISH *imputed* genes are predicted from Allen scRNA, so they
are tracked as *supporting* evidence only — not counted toward independent
concordance. Run notebooks 01–04 first so the aggregate parquets exist (re-run them
after the recent code changes so they carry `frac_expressing`/`n_cells`).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
import pandas as pd

from src.config import load_config, resolve_output_dir, restrict_config_to_genes, start_run
from src.data_loaders import get_abc_cache, merfish_gene_source_map
from src import synthesis as syn
from src.utils import print_path

In [ ]:
# Unified panel (receptors + excitability). Synthesis carries the 'category' column.
CONFIG_PATH = PROJECT_ROOT / "query_config.yaml"
config = load_config(CONFIG_PATH)
EXPLORATION_ROOT = resolve_output_dir(cfg=config)
config["_dataset_modality"] = "synthesis"

OUTPUT_DIR = start_run(
    PROJECT_ROOT,
    config,
    dataset="synthesis",
    exploration_root=EXPLORATION_ROOT,
    notebook="05_synthesis",
)
assert not str(OUTPUT_DIR).startswith(str(PROJECT_ROOT)), (
    f"OUTPUT_DIR must be outside the repo; got {OUTPUT_DIR}"
)

print(f"Gene panel key: {config['_gene_panel_key']}")
print(f"Genes: {len(config['_all_genes'])}")
print(f"Brain areas: {config['brain_areas']}")
print(f"Cell type level: {config['cell_type_level']}")
print(f"Run dir: {OUTPUT_DIR}")

In [ ]:
cache = get_abc_cache(config)
print("Manifest:", cache.current_manifest)

aggregates, sources = syn.gather_dataset_aggregates(
    config, exploration_root=EXPLORATION_ROOT,
)
print(f"\nDatasets loaded: {sorted(aggregates)}")
for key, df in aggregates.items():
    has_frac = "frac_expressing" in df.columns
    print(f"  {key}: {len(df):,} rows, frac_expressing={'yes' if has_frac else 'MISSING (re-run nb)'}")
    print(f"    from {sources.get(key)}")

In [ ]:
# Allen MERFISH measured-vs-imputed provenance (drives independence accounting).
allen_sources = {}
if "allen_merfish" in aggregates:
    genes_present = sorted(aggregates["allen_merfish"]["gene"].unique())
    allen_sources = merfish_gene_source_map(cache, genes_present, config)
    n_imp = sum(1 for v in allen_sources.values() if v == "imputed")
    print(f"Allen MERFISH genes: {len(allen_sources)} ({n_imp} imputed, supporting-only)")

evidence = syn.build_evidence_table(
    aggregates, config, allen_gene_sources=allen_sources,
)
print(f"\nEvidence rows (cell_type × region × gene): {len(evidence):,}")
print("\nConfidence tier counts:")
print(evidence["confidence_tier"].value_counts())

In [ ]:
if config["output"].get("save_processed_data", True):
    pq = OUTPUT_DIR / "evidence_table.parquet"
    csv = OUTPUT_DIR / "evidence_table.csv"
    evidence.to_parquet(pq, index=False)
    evidence.to_csv(csv, index=False)
    print(f"Saved {pq}")
    print(f"Saved {csv}")

## Focused statement for a target (cell_type, region)

Set `TARGET_CELL_TYPE` to a value present at the configured `cell_type_level`
(e.g. a supertype/subclass/cluster name) and `TARGET_REGION` to a config brain
area. Leave either as `None` to pool. The biologically relevant level/clusters
are chosen here — the pipeline itself is level-agnostic.

In [ ]:
# Inspect available cell types at this level (pick one for the target below).
print(sorted(evidence["cell_type"].unique())[:40])

In [ ]:
TARGET_CELL_TYPE = None  # e.g. a name containing 'L5 ET'
TARGET_REGION = "VISpm"  # or None to pool across regions

if TARGET_CELL_TYPE is None:
    # Default: the cell type with the most high-confidence genes in TARGET_REGION.
    scope = evidence if TARGET_REGION is None else evidence[evidence["brain_area"] == TARGET_REGION]
    high = scope[scope["confidence_tier"] == "high"]
    if not high.empty:
        TARGET_CELL_TYPE = high["cell_type"].value_counts().idxmax()
        print(f"Auto-selected TARGET_CELL_TYPE = {TARGET_CELL_TYPE!r}")
    else:
        raise RuntimeError("No high-confidence rows; set TARGET_CELL_TYPE manually.")

summary = syn.summarize_target(evidence, cell_type=TARGET_CELL_TYPE, region=TARGET_REGION)
print(syn.statement_scaffold(summary))

In [ ]:
tev = syn.target_evidence(evidence, cell_type=TARGET_CELL_TYPE, region=TARGET_REGION)
target_cols = [
    "gene", "family", "confidence_tier",
    "n_independent_measured_detections", "supporting_imputed_detection",
]
display(tev[target_cols].head(40))

dot = syn.plot_evidence_dotplot(
    evidence, config,
    cell_type=TARGET_CELL_TYPE, region=TARGET_REGION,
    output_dir=OUTPUT_DIR,
)
print(f"Saved {dot}")